### Nama: Yohanes Gerardus Haga Zai
### Tema analisis: rekomendasi area/daerah bagi wisatawan ataupun penduduk lokal untuk liburan

#### 1. Import library

In [8]:
import requests
import folium
import geopandas as gpd
from shapely.geometry import shape, MultiPolygon
from pyproj import CRS
import json
import os

#### 2. Lengkapi URL data
##### Variabel atau data yang digunakan antara lain
* Wilaya terancam tsunami
* Wilaya resiko banjir
* Restoran
* Makanan dan minuman
* Fasilitas olahraga
* Area Lapangan
* Wisata alam
* Parawisata

In [9]:
urls = {
  "WilayaTerancamTsunami": os.environ.get("KeyData_WilayaTerancamTsunami"),
  "WilayaResikoBanjir": os.environ.get("KeyData_WilayaResikoBanjir"),
  "Restoran": os.environ.get("KeyData_Restoran"),
  "MakananMinuman": os.environ.get("KeyData_MakananMinuman"),
  "FasilitasOlahraga": os.environ.get("KeyData_FasilitasOlahraga"),
  "AreaLapangan": os.environ.get("KeyData_AreaLapangan"),
  "WisataAlam": os.environ.get("KeyData_WisataAlam"),
  "Parawisata": os.environ.get("KeyData_Parawisata")
}

#### 3. Membaca dan mengonversi data GeoJson Ke GeoDataFrame

In [10]:
def get_gdf_from_api(url):
    response = requests.get(url)
    if response.status_code == 200:
        geojson = response.json()
        gdf = gpd.GeoDataFrame.from_features(geojson['features'])
        print(f"Berhasil mengambil data dari {url.split('&layer_id=')[1][:10]}...")
        return gdf
    else:
        print(f"Gagal mengambil data dari {url}")
        return gpd.GeoDataFrame()

# Membaca semua data ke dalam dictionary GeoDataFrame
gdfs = {}
for key, url in urls.items():
    gdfs[key] = get_gdf_from_api(url)

Berhasil mengambil data dari 69a26ffe85...
Berhasil mengambil data dari 69a2702064...
Berhasil mengambil data dari 69a2711164...
Berhasil mengambil data dari 69a2717185...
Berhasil mengambil data dari 69a274136c...
Berhasil mengambil data dari 69a274446c...
Berhasil mengambil data dari 69a274786c...
Berhasil mengambil data dari 69a2748185...


#### 4. Pemeriksaan & Penyesuaian Data (Null & CRS)

In [11]:
for key, gdf in gdfs.items():
    # Isi null dengan default (0 atau 'TIDAK DIKETAHUI')
    for col in gdf.columns:
        if gdf[col].dtype == 'O':
            gdf[col] = gdf[col].fillna('TIDAK DIKETAHUI')
        else:
            gdf[col] = gdf[col].fillna(0)
    # Set CRS ke EPSG:4326 jika belum
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf.set_crs(epsg=4326, inplace=True)
    gdfs[key] = gdf
print("Semua data sudah dicek dan disesuaikan.")

Semua data sudah dicek dan disesuaikan.


In [12]:
for key, gdf in gdfs.items():
    # Pastikan GeoDataFrame tidak kosong dan bertipe Point
    if not gdf.empty and (gdf.geom_type == 'Point').all():
        # Buffer otomatis menghasilkan Polygon
        gdf['geometry'] = gdf.geometry.buffer(0.001)
        
        # 2. Bungkus ke MultiPolygon jika memang diwajibkan format MultiPolygon
        gdf['geometry'] = gdf['geometry'].apply(lambda x: MultiPolygon([x]) if x else None)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20888\101580791.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['geometry'] = gdf.geometry.buffer(0.001)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_20888\101580791.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['geometry'] = gdf.geometry.buffer(0.001)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_20888\101580791.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['geometry'] = gdf.geometry.buffer(0.001)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_20888\101580791.py:5: UserWarning: Geometry is in a geographic CRS. Resu

#### 4. Menampilkan peta setiap variabel menggunakan Folium untuk eksplorasi awal.

In [13]:
for key, gdf in gdfs.items():
    m = folium.Map(location=[3.59, 98.67], zoom_start= 11)
    folium.GeoJson(gdf).add_to(m)
    folium.LayerControl().add_to(m)
    print(f"Peta di bawah ini, untuk variabel: {key}")
    display(m)  

Peta di bawah ini, untuk variabel: WilayaTerancamTsunami


Peta di bawah ini, untuk variabel: WilayaResikoBanjir


Peta di bawah ini, untuk variabel: Restoran


Peta di bawah ini, untuk variabel: MakananMinuman


Peta di bawah ini, untuk variabel: FasilitasOlahraga


Peta di bawah ini, untuk variabel: AreaLapangan


Peta di bawah ini, untuk variabel: WisataAlam


Peta di bawah ini, untuk variabel: Parawisata


#### 5. Overlay/Intersect Data Spasial Multi-Variabel
##### Menggabungkan seluruh layer spasial berdasarkan geometri yang saling tumpang tindih (intersect) untuk menghasilkan satu GeoDataFrame komposit.

In [14]:
# Pastikan kolom 'ID' dihapus/rename agar tidak error saat overlay
for key in gdfs.keys():
    if 'ID' in gdfs[key].columns:
        gdfs[key] = gdfs[key].drop(columns=['ID'])

# List GeoDataFrame yang akan di-intersect (urutkan sesuai kebutuhan analisis)
gdf_list = [
    gdfs['WilayaTerancamTsunami'],
    gdfs['WilayaResikoBanjir'],
    gdfs['Restoran'],
    gdfs['Parawisata'],
    gdfs['FasilitasOlahraga'],
    gdfs['MakananMinuman'],
    gdfs['AreaLapangan'],
    gdfs['WisataAlam'],
]

# Fungsi overlay/intersect berurutan
def overlay_multiple(gdf_list):
    result = gdf_list[0]
    for gdf in gdf_list[1:]:
        if not gdf.empty:
            result = gpd.overlay(result, gdf, how='intersection')
            result = result[result.is_valid]
    return result

gdf_intersect = overlay_multiple(gdf_list)
print(f"Jumlah area hasil intersect: {len(gdf_intersect)}")

MergeError: Passing 'suffixes' which cause duplicate columns {'KECAMATAN_1', 'ID_PROV_1', 'DESA_1', 'KABKOT_1', 'ID_KABKOT_1', 'ID_DESA_1', 'PROVINSI_1', 'ID_KEC_1'} is not allowed.